# Stage 2b pilot instrument: does the specific fitted map matter?

Observation only. This notebook ships with both authorization flags false and therefore refuses before CUDA/model loading. It supports the smallest auditable pilot path only; confirmatory execution remains a later, separately authorized stage after pilot thresholds are locked.

Scientific boundaries: no fitting, steering, ablation, activation editing, or causal intervention. The fitted/broken-map and correct/wrong-activation cells are offline readouts. Evidence remains class 1. The producer retains factorized raw scores, derives the ratified denominator guard after all 80 loci, computes both uncertainty methods, and emits threshold-derivation evidence without a pilot decision.

The model-output argmax target is **NOT delegable**: it measures movement toward the model's own next-token trajectory, not truth or task correctness. The decoded input embedding is a weak floor, so absolute NTA curves are **DESCRIPTIVE ONLY**.


## 1. Pinned identities, partition, and authorization


In [ ]:
import hashlib
import json
import os
import pathlib

MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
EXPECTED_MODEL_D_MODEL = 2048
EXPECTED_MODEL_N_LAYERS = 28
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "a4114d7752d11eb546e6cf372213d7e75526d3a1"
LENS_FILE = "qwen3-1.7b/jlens/Salesforce-wikitext/Qwen3-1.7B_jacobian_lens.pt"
EXPECTED_LENS_SHA256 = "6fcc79011bd921ffd87612255e2e99950a124fa519470ee44ebaf161c39be9d6"
JLENS_REPO_URL = "https://github.com/anthropics/jacobian-lens.git"
JLENS_COMMIT = "581d398613e5602a5af361e1c34d3a92ea82ba8e"
EXPECTED_RUNTIME_VERSIONS = {
    "transformers": "5.5.4",
    "huggingface_hub": "1.24.0",
    "numpy": "2.5.1",
    "scipy": "1.18.0",
    "safetensors": "0.8.0",
    "accelerate": "1.14.0",
    "torch": "2.13.0",
}
RUNTIME_INSTALL_SCHEMA = "stage2b-colab-runtime-install/v2"
RUNTIME_INSTALL_SENTINEL = pathlib.Path("/content/stage2b_pilot_runtime_install.json")
RUNTIME_REMOVE_PACKAGES = ("torchvision",)
RUNTIME_INSTALL_REQUIREMENTS = (
    f"git+{JLENS_REPO_URL}@{JLENS_COMMIT}",
    f"transformers=={EXPECTED_RUNTIME_VERSIONS['transformers']}",
    f"huggingface_hub=={EXPECTED_RUNTIME_VERSIONS['huggingface_hub']}",
    f"safetensors=={EXPECTED_RUNTIME_VERSIONS['safetensors']}",
    f"scipy=={EXPECTED_RUNTIME_VERSIONS['scipy']}",
    f"numpy=={EXPECTED_RUNTIME_VERSIONS['numpy']}",
    f"accelerate=={EXPECTED_RUNTIME_VERSIONS['accelerate']}",
    f"torch=={EXPECTED_RUNTIME_VERSIONS['torch']}",
)
runtime_install_spec = {
    "schema": RUNTIME_INSTALL_SCHEMA,
    "remove_packages": list(RUNTIME_REMOVE_PACKAGES),
    "requirements": list(RUNTIME_INSTALL_REQUIREMENTS),
    "expected_runtime_versions": EXPECTED_RUNTIME_VERSIONS,
}
runtime_install_spec_payload = (
    json.dumps(runtime_install_spec, sort_keys=True, separators=(",", ":")) + "\n"
).encode("utf-8")
install_spec_sha256 = hashlib.sha256(runtime_install_spec_payload).hexdigest()
PROCESS_IDENTITY = (
    f"{os.getpid()}:"
    f"{pathlib.Path('/proc/self/stat').read_text(encoding='utf-8').split()[21]}"
)
EXPECTED_MANIFEST_SHA256 = "ba29c629c7b9601980b6c0bb9cd9730242d7cd6b7eacb1166c307837416d4bbf"

SELECTED_LAYERS = [6, 13, 20, 26]
POSITIONS = [-2]
MAX_PROMPT_TOKENS = 128
MIN_VRAM_GIB = 14.0
TOP_K = 10
SCHEMA = "jspace-observation-stage2b/v1"

RUN_MODE = "pilot"
PILOT_AUTHORIZED = False
PILOT_PROTOCOL_RATIFIED = False
THRESHOLDS_RATIFIED = False
ARTIFACT_TRANSFER_AUTHORIZED = False

SPEC_MIN_EFFECT = None
NTA_MIN_DENOMINATOR = None
INTERACTION_MIN_EFFECT = None
PROMPT_ONLY_CONSTRUCTION = "input_embedding_decoded"
NONREDUNDANCY_MAX_JACCARD = None
DECODE_PARITY_TOL = 1e-5
STAGE1_RERUN_NOISE_MAX_ABS_LOGIT_DIFF = 0.0
BROKEN_MAP_DRAWS = None  # supplied only by the ratified registry after authorization
WRONG_ACTIVATION_ASSIGNMENTS = None  # supplied only by the ratified registry after authorization
STAGE1_PROMPT_SHA256 = "daeaa63881dc0f58be689307a81b1fbc347674424f1cae45819f82372804f5a6"
STAGE2B_N_PROMPTS = 200
STAGE2B_N_CATEGORIES = 5

# Ratified 20-prompt pilot identity. Authorization remains a separate gate.
PILOT_IDS = [
    "s000", "s001", "s002", "s003",
    "s040", "s041", "s042", "s043",
    "s080", "s081", "s082", "s083",
    "s120", "s121", "s122", "s123",
    "s160", "s161", "s162", "s163",
]
EXPECTED_PILOT_SUBSET_SHA256 = "8ed0a0092ec3989f6bd8005ae4360de86174764a946af75b35ea30932ca719b5"
EXPECTED_PILOT_VIEW_SHA256 = "5bef8316f72682a628fc1240bf6068a91aa7c8a330377206cbd9145434b797e4"


## 2. Load tested modules and fail at the authorization boundary

The gate is deliberately before package installation, CUDA inspection, model downloads, and model loading.


In [ ]:
import hashlib
import json
import pathlib
import sys
import tempfile
import zipfile

PILOT_CODE_BUNDLE = pathlib.Path("stage2b-pilot-code-bundle.zip")
if not PILOT_CODE_BUNDLE.is_file():
    raise FileNotFoundError(f"required code bundle is missing: {PILOT_CODE_BUNDLE}")
OBSERVED_CODE_BUNDLE_SHA256 = hashlib.sha256(PILOT_CODE_BUNDLE.read_bytes()).hexdigest()
BUNDLE_ROOT = pathlib.Path(tempfile.mkdtemp(prefix="stage2b-pilot-code-"))
with zipfile.ZipFile(PILOT_CODE_BUNDLE) as archive:
    members = archive.infolist()
    if not members or any(
        pathlib.PurePosixPath(member.filename).is_absolute()
        or ".." in pathlib.PurePosixPath(member.filename).parts
        or member.is_dir()
        or ((member.external_attr >> 16) & 0o170000) == 0o120000
        for member in members
    ):
        raise RuntimeError("pilot code bundle contains an unsafe member path")
    BUNDLE_MANIFEST = json.loads(archive.read("bundle-manifest.json"))
    if BUNDLE_MANIFEST.get("schema") != "jspace-stage2b-pilot-code-bundle/v1":
        raise RuntimeError("pilot code bundle manifest schema is invalid")
    declared_files = BUNDLE_MANIFEST.get("files")
    if not isinstance(declared_files, list):
        raise RuntimeError("pilot code bundle manifest file list is invalid")
    expected_members = [entry.get("name") for entry in declared_files] + ["bundle-manifest.json"]
    if archive.namelist() != expected_members or len(set(expected_members)) != len(expected_members):
        raise RuntimeError("pilot code bundle member set differs from its manifest")
    archive.extractall(BUNDLE_ROOT)
actual_files = sorted(
    str(path.relative_to(BUNDLE_ROOT).as_posix())
    for path in BUNDLE_ROOT.rglob("*")
    if path.is_file()
)
if actual_files != sorted(expected_members):
    raise RuntimeError("pilot code extraction produced an unexpected file set")
for entry in declared_files:
    extracted = BUNDLE_ROOT / entry["name"]
    payload = extracted.read_bytes()
    if len(payload) != entry.get("size_bytes") or hashlib.sha256(payload).hexdigest() != entry.get("sha256"):
        raise RuntimeError(f"pilot code extraction identity mismatch: {entry['name']}")
SCRIPTS = BUNDLE_ROOT / "EvoScientist/skills/jspace-research-operations/scripts"
if not SCRIPTS.exists():
    raise FileNotFoundError(f"tested Stage 2b modules are absent from {PILOT_CODE_BUNDLE}")
sys.path.insert(0, str(SCRIPTS))

import stage2b_endpoint as ep  # noqa: E402 - path is established above
import stage2b_manifest as mf  # noqa: E402 - path is established above
import stage2b_preflight as pf  # noqa: E402 - path is established above
import stage2b_statistics as st  # noqa: E402 - path is established above
import validate_observation  # noqa: E402 - path is established above

APPROVED_AUTHORIZATION_RECORD_SHA256 = input(
    "Paste the independently approved Stage 2b pilot authorization SHA-256: "
).strip()
authorization_path = pathlib.Path(
    f"stage2b-pilot-authorization-{APPROVED_AUTHORIZATION_RECORD_SHA256}.json"
)
AUTHORIZATION_RECORD = pf.load_pilot_authorization_record(
    authorization_path,
    approved_record_sha256=APPROVED_AUTHORIZATION_RECORD_SHA256,
    expected_pilot_view_sha256=EXPECTED_PILOT_VIEW_SHA256,
    observed_code_bundle_sha256=OBSERVED_CODE_BUNDLE_SHA256,
)
AUTHORIZATION_RECORD_SHA256 = AUTHORIZATION_RECORD["_record_sha256"]
TRUSTED_SOURCE_IDENTITIES = {
    "authorization_record_sha256": AUTHORIZATION_RECORD_SHA256,
    "notebook_sha256": AUTHORIZATION_RECORD["source"]["notebook_sha256"],
    "code_bundle_sha256": AUTHORIZATION_RECORD["source"]["code_bundle_sha256"],
}
AUTHORIZATION, REGISTRY = pf.materialize_pilot_authorization(
    AUTHORIZATION_RECORD, pf.INITIAL_REGISTRY
)
PILOT_AUTHORIZED = AUTHORIZATION["PILOT_AUTHORIZED"]
PILOT_PROTOCOL_RATIFIED = AUTHORIZATION["PILOT_PROTOCOL_RATIFIED"]
THRESHOLDS_RATIFIED = AUTHORIZATION["THRESHOLDS_RATIFIED"]
ARTIFACT_TRANSFER_AUTHORIZED = AUTHORIZATION_RECORD["scope"]["artifact_transfer_authorized"]
BROKEN_MAP_DRAWS = AUTHORIZATION["BROKEN_MAP_DRAWS"]
WRONG_ACTIVATION_ASSIGNMENTS = AUTHORIZATION["WRONG_ACTIVATION_ASSIGNMENTS"]
pf.check_constant_registry(REGISTRY, pf.GATES)

pf.check_ratification(AUTHORIZATION, REGISTRY, mode=RUN_MODE)
pf.check_crossing_registry(WRONG_ACTIVATION_ASSIGNMENTS, BROKEN_MAP_DRAWS)
if RUN_MODE != "pilot":
    raise RuntimeError("this isolated notebook is pilot-only; confirmation needs its own holdout view")
if REGISTRY["NTA_MIN_DENOMINATOR"]["declared_value"] is not None:
    raise RuntimeError("pilot authorization must not inject the data-derived denominator guard")
print(f"authorized run mode: {RUN_MODE}")


## 3. Install pinned runtime dependencies


In [ ]:
import subprocess

runtime_remove_command = [
    sys.executable, "-m", "pip", "uninstall", "-y", *RUNTIME_REMOVE_PACKAGES
]
runtime_install_command = [
    sys.executable, "-m", "pip", "install", "-q", *RUNTIME_INSTALL_REQUIREMENTS
]
if RUNTIME_INSTALL_SENTINEL.is_file():
    install_record = json.loads(RUNTIME_INSTALL_SENTINEL.read_text(encoding="utf-8"))
    if set(install_record) != {
        "schema", "install_spec_sha256", "install_process_identity"
    }:
        raise RuntimeError("runtime install sentinel has unexpected fields")
    if install_record["schema"] != RUNTIME_INSTALL_SCHEMA:
        raise RuntimeError("runtime install sentinel schema mismatch")
    if install_record["install_spec_sha256"] != install_spec_sha256:
        raise RuntimeError("runtime install specification SHA-256 mismatch")
    fresh_process_after_install = (
        install_record["install_process_identity"] != PROCESS_IDENTITY
    )
    if fresh_process_after_install:
        print("fresh Colab Python process verified after pinned installation")
    else:
        print("pinned installation is complete; restart the Colab session")
else:
    subprocess.run(runtime_remove_command, check=True)
    subprocess.run(runtime_install_command, check=True)
    install_record = {
        "schema": RUNTIME_INSTALL_SCHEMA,
        "install_spec_sha256": install_spec_sha256,
        "install_process_identity": PROCESS_IDENTITY,
    }
    install_record_payload = (
        json.dumps(install_record, sort_keys=True, indent=2) + "\n"
    ).encode("utf-8")
    with RUNTIME_INSTALL_SENTINEL.open("xb") as handle:
        handle.write(install_record_payload)
    if RUNTIME_INSTALL_SENTINEL.read_bytes() != install_record_payload:
        raise RuntimeError("runtime install sentinel readback mismatch")
    fresh_process_after_install = False
    print("pinned installation complete; use Runtime > Restart session")


## 4. Verify installed identity, manifest, partition, and capacity before model loading


In [ ]:
if not RUNTIME_INSTALL_SENTINEL.is_file():
    raise RuntimeError("runtime install sentinel is missing")
install_record = json.loads(RUNTIME_INSTALL_SENTINEL.read_text(encoding="utf-8"))
if install_record.get("install_spec_sha256") != install_spec_sha256:
    raise RuntimeError("runtime install specification SHA-256 mismatch")
fresh_process_after_install = (
    install_record.get("install_process_identity") != PROCESS_IDENTITY
)
if not fresh_process_after_install:
    raise RuntimeError("runtime restart required before package imports")

import importlib.metadata
import importlib.util

torchvision_distribution = None
try:
    torchvision_distribution = importlib.metadata.version("torchvision")
except importlib.metadata.PackageNotFoundError:
    pass
if (
    torchvision_distribution is not None
    or importlib.util.find_spec("torchvision") is not None
):
    raise RuntimeError("torchvision must be absent from the text-only runtime")
torchvision_state = "absent"

import huggingface_hub
import numpy as np
import torch
import transformers

observed_runtime_versions = {
    name: importlib.metadata.version(name.replace("_", "-"))
    for name in EXPECTED_RUNTIME_VERSIONS
}
assert observed_runtime_versions == EXPECTED_RUNTIME_VERSIONS

try:
    import jlens
except ImportError as exc:
    raise RuntimeError("jlens is not installed; run the install cell") from exc

distribution = importlib.metadata.distribution("jlens")
installed_commit = pf.installed_vcs_commit(
    distribution.read_text("direct_url.json"),
    expected_repo_url=JLENS_REPO_URL,
)

MANIFEST_PATH = pathlib.Path("jspace-stage2b-pilot-v1.json")
if hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest() != EXPECTED_PILOT_VIEW_SHA256:
    raise RuntimeError("uploaded pilot-view SHA-256 mismatch")
manifest = json.loads(MANIFEST_PATH.read_text())
stage2_digests = json.loads(
    (BUNDLE_ROOT / "tests/jspace/fixtures/stage2_manifest_digests.json").read_text()
)["digests"]
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION
)
token_counts = {
    prompt["text"]: len(tokenizer(prompt["text"])["input_ids"])
    for prompt in manifest["prompts"]
}
mf.check_pilot_view(
    manifest,
    stage2_digests,
    expected_view_digest=EXPECTED_PILOT_VIEW_SHA256,
    expected_source_manifest_digest=EXPECTED_MANIFEST_SHA256,
    expected_source_n_prompts=STAGE2B_N_PROMPTS,
    expected_n_categories=STAGE2B_N_CATEGORIES,
    expected_pilot_subset_digest=EXPECTED_PILOT_SUBSET_SHA256,
    expected_pilot_ids=PILOT_IDS,
    token_counts=token_counts,
    max_prompt_tokens=MAX_PROMPT_TOKENS,
    stage1_anchor_sha256=STAGE1_PROMPT_SHA256,
)
partition = {
    "n_prompts": manifest["n_prompts"],
    "pilot_subset_sha256": manifest["pilot_subset_sha256"],
}

env = {
    "python_version": tuple(sys.version_info[:3]),
    "cuda_available": torch.cuda.is_available(),
    "vram_gib": (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        if torch.cuda.is_available()
        else 0.0
    ),
    "jlens_commit": installed_commit,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "lens_repo": LENS_REPO,
    "lens_revision": LENS_REVISION,
    "lens_file": LENS_FILE,
    "expected_lens_sha256": EXPECTED_LENS_SHA256,
    "expected_model_d_model": EXPECTED_MODEL_D_MODEL,
    "expected_model_n_layers": EXPECTED_MODEL_N_LAYERS,
}
PINS = {name: entry["declared_value"] for name, entry in REGISTRY.items()}
pf.check_environment(env, PINS)
print(f"preflight passed for {partition['n_prompts']} {RUN_MODE} prompts")


## 5. Load the model and fitted lens; verify direct pinned APIs and tensor contracts


In [ ]:
from jlens.hooks import ActivationRecorder
from jlens.vis import _ranks_of

model_hf = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
tokenizer.padding_side = "right"
model = jlens.from_hf(model_hf, tokenizer, compile=False)
lens_path = huggingface_hub.hf_hub_download(
    repo_id=LENS_REPO, revision=LENS_REVISION, filename=LENS_FILE
)
assert hashlib.sha256(pathlib.Path(lens_path).read_bytes()).hexdigest() == EXPECTED_LENS_SHA256
lens = jlens.JacobianLens.load(lens_path)

assert hasattr(lens, "jacobians")
assert callable(getattr(lens, "transport", None))
assert set(SELECTED_LAYERS) <= set(lens.source_layers)
for layer in SELECTED_LAYERS:
    assert tuple(lens.jacobians[layer].shape) == (
        EXPECTED_MODEL_D_MODEL, EXPECTED_MODEL_D_MODEL
    )
    assert str(lens.jacobians[layer].dtype) == "torch.float32"


def capture_prompt(text):
    input_ids = model.encode(text, max_length=MAX_PROMPT_TOKENS)
    final_layer = model.n_layers - 1
    record_at = sorted(set(SELECTED_LAYERS) | {0, final_layer})
    with torch.inference_mode(), ActivationRecorder(model.layers, at=record_at) as recorder:
        model.forward(input_ids)
    residuals = {
        layer: recorder.activations[layer][0, POSITIONS[0], :].detach().float().cpu()
        for layer in SELECTED_LAYERS
    }
    layer0_residual = recorder.activations[0][0, POSITIONS[0], :].detach().float().cpu()
    final_residual = recorder.activations[final_layer][0, POSITIONS[0], :].detach().float()
    output_logits = model.unembed(final_residual).float().cpu()
    return residuals, layer0_residual, output_logits, input_ids


def decode_numpy(vector):
    tensor = torch.as_tensor(vector, dtype=torch.float32, device=model.input_device)
    return model.unembed(tensor).float().cpu()


probe_text = manifest["prompts"][0]["text"]
probe_residuals, _, _, _ = capture_prompt(probe_text)
probe_layer = SELECTED_LAYERS[0]
probe_residual = probe_residuals[probe_layer]
baseline_logits, _, _ = lens.apply(
    model,
    probe_text,
    layers=[probe_layer],
    positions=POSITIONS,
    max_seq_len=MAX_PROMPT_TOKENS,
    use_jacobian=False,
)
direct_logits = decode_numpy(probe_residual.numpy())
decode_parity_max_abs = float(
    (direct_logits - baseline_logits[probe_layer][0]).abs().max()
)
np_transport = ep.transport_with(
    probe_residual.numpy(), lens.jacobians[probe_layer].numpy()
)
tl_transport = lens.transport(probe_residual, probe_layer).cpu().numpy()
assert np.allclose(np_transport, tl_transport, rtol=1e-5, atol=1e-5)

# Direct parity against the pinned jlens implementation on unique logits.
# Ties retain Stage 2b's separately preregistered best-rank convention.
rank_probe_logits = torch.tensor([[0.1, 0.4, -0.2, 0.3]], dtype=torch.float32)
rank_probe_targets = torch.tensor([0, 1, 2, 3], dtype=torch.long)
jlens_rank1 = (_ranks_of(rank_probe_logits, rank_probe_targets) + 1)[0].tolist()
rank_parity_verified = all(
    ep.target_rank1(rank_probe_logits[0].tolist(), int(target_id)) == reference_rank
    for target_id, reference_rank in zip(rank_probe_targets, jlens_rank1, strict=True)
)
assert rank_parity_verified

TENSOR_CONTRACT = {
    "residual_shape": tuple(probe_residual.shape),
    "residual_dtype": str(probe_residual.dtype),
    "jacobian_shape": tuple(lens.jacobians[probe_layer].shape),
    "jacobian_dtype": str(lens.jacobians[probe_layer].dtype),
    "readout_device": str(direct_logits.device),
    "decode_parity_max_abs": decode_parity_max_abs,
    "decode_parity_tol": DECODE_PARITY_TOL,
    "logit_softcapping": getattr(model, "_logit_softcap", None),
    "rank_parity_verified": rank_parity_verified,
    "primary_floor_id": "input_embedding_decoded",
    "sensitivity_floor_id": "layer0_residual_decoded",
}
pf.check_tensor_contracts(TENSOR_CONTRACT, d_model=EXPECTED_MODEL_D_MODEL)
print("direct transport and tensor contracts passed")

## 6. Capture the authorized partition and execute the full crossed readout path

No prompt text, raw activation, or full-vocabulary logits are persisted. The producer consumes only the ratified donor/map registries. It retains all normalized scores, derives the one run-wide denominator guard after all 80 loci, computes both floor trees and both ratified uncertainty procedures without another model/lens pass, and emits threshold evidence without a decision.


In [ ]:
import uuid

# Q3 is NOT delegable: model argmax measures model-trajectory fidelity only.
# Absolute NTA against the weak embedding floor is DESCRIPTIVE ONLY.
RUN_ID = str(uuid.uuid4())
V = int(model_hf.config.vocab_size)
residuals_by_layer = {layer: {} for layer in SELECTED_LAYERS}
layer0_residuals = {}
outputs_by_prompt = {}

for meta in manifest["prompts"]:
    captured, layer0_residual, output_logits, _ = capture_prompt(meta["text"])
    digest = meta["sha256"]
    outputs_by_prompt[digest] = output_logits
    layer0_residuals[digest] = layer0_residual.numpy()
    for layer, residual in captured.items():
        residuals_by_layer[layer][digest] = residual.numpy()

def array_sha256(value):
    array = np.ascontiguousarray(value)
    metadata = f"{array.dtype}:{array.shape}:".encode("ascii")
    return hashlib.sha256(metadata + array.tobytes()).hexdigest()


def map_factorized(source, transform):
    return {
        "correct_act_fitted_map": transform(source["correct_act_fitted_map"]),
        "correct_act_broken_map": {
            map_id: transform(value)
            for map_id, value in source["correct_act_broken_map"].items()
        },
        "wrong_act_fitted_map": {
            donor_id: transform(value)
            for donor_id, value in source["wrong_act_fitted_map"].items()
        },
        "wrong_act_broken_map": {
            donor_id: {map_id: transform(value) for map_id, value in row.items()}
            for donor_id, row in source["wrong_act_broken_map"].items()
        },
    }

broken_maps_by_layer = {}
map_draws_by_layer = {}
for layer in SELECTED_LAYERS:
    fitted = lens.jacobians[layer].numpy()
    broken_maps = {}
    map_draws = []
    realized_maps, fitted_singular_values = ep.build_fit_broken_maps(
        fitted, [entry["seed"] for entry in BROKEN_MAP_DRAWS]
    )
    for map_entry, broken_map in zip(BROKEN_MAP_DRAWS, realized_maps, strict=True):
        map_id = map_entry["id"]
        seed = map_entry["seed"]
        spectrum_check = ep.singular_spectrum_evidence(
            fitted, broken_map, fitted_singular_values=fitted_singular_values
        )
        if not spectrum_check["verified"]:
            raise RuntimeError(f"broken-map spectrum verification failed: layer={layer} map={map_id}")
        broken_maps[map_id] = broken_map
        map_draws.append({
            "map_draw_id": map_id,
            "seed_index": map_entry["index"],
            "seed_namespace": map_entry["namespace"],
            "seed_sha256": map_entry["sha256"],
            "seed": seed,
            "bit_generator": map_entry["bit_generator"],
            "sha256": array_sha256(broken_map),
            "spectrum_check": spectrum_check,
        })
    broken_maps_by_layer[layer] = broken_maps
    map_draws_by_layer[layer] = map_draws

raw_records = []
for meta in manifest["prompts"]:
    digest = meta["sha256"]
    output_logits = outputs_by_prompt[digest]
    max_logit = output_logits.max()
    argmax_tie_token_ids = torch.nonzero(
        output_logits == max_logit, as_tuple=False
    ).flatten().tolist()
    target_id = min(argmax_tie_token_ids)
    target_derivation = {
        "method": "model_argmax",
        "output_logits_sha256": array_sha256(output_logits.numpy()),
        "output_logits_dtype": str(output_logits.numpy().dtype),
        "output_logits_shape": list(output_logits.shape),
        "max_logit": float(max_logit),
        "argmax_tie_token_ids": argmax_tie_token_ids,
        "tie_break_rule": "lowest_token_id",
        "runtime_verifier_id": validate_observation.STAGE2B_TARGET_RUNTIME_VERIFIER,
        "runtime_verified": True,
    }
    target_derivation["target_decision_sha256"] = ep.target_decision_sha256(
        target_id, target_derivation
    )
    target_verification_errors = validate_observation.verify_target_derivation_against_logits(
        output_logits.numpy(), target_id, target_derivation
    )
    if target_verification_errors:
        raise RuntimeError("target derivation runtime verification failed: " + "; ".join(target_verification_errors))
    prompt_ids = tokenizer(
        meta["text"], return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS
    ).input_ids.to(model.input_device)
    prompt_embedding = model_hf.get_input_embeddings()(prompt_ids)[0, POSITIONS[0], :]
    prompt_logits = model.unembed(prompt_embedding).float().cpu()
    layer0_logits = decode_numpy(layer0_residuals[digest])

    def score(logits):
        rank = ep.target_rank1(logits.tolist(), target_id)
        return {"rank1": rank, "s": ep.rank_score(rank, V)}

    output_readout = score(output_logits)
    primary_floor = score(prompt_logits)
    sensitivity_floor = score(layer0_logits)

    for layer in SELECTED_LAYERS:
        correct = residuals_by_layer[layer][digest]
        fitted = lens.jacobians[layer].numpy()
        broken_maps = broken_maps_by_layer[layer]
        wrong_residuals = {}
        donor_assignments = []
        for donor_entry in WRONG_ACTIVATION_ASSIGNMENTS:
            donor_id = donor_entry["id"]
            seed = donor_entry["seed"]
            wrong, donor_digest = ep.select_wrong_activation(
                residuals_by_layer[layer], digest, seed=seed
            )
            wrong_residuals[donor_id] = wrong
            donor_assignments.append({
                "donor_assignment_id": donor_id,
                "seed_index": donor_entry["index"],
                "seed_namespace": donor_entry["namespace"],
                "seed_sha256": donor_entry["sha256"],
                "seed": seed,
                "bit_generator": donor_entry["bit_generator"],
                "recipient_prompt_sha256": digest,
                "source_prompt_sha256": donor_digest,
                "recipient_to_donor_sha256": hashlib.sha256(
                    f"{digest}->{donor_digest}".encode("ascii")
                ).hexdigest(),
                "residual_sha256": array_sha256(wrong),
            })

        vectors = {
            "correct_act_fitted_map": ep.transport_with(correct, fitted),
            "correct_act_broken_map": {
                map_id: ep.transport_with(correct, broken_map)
                for map_id, broken_map in broken_maps.items()
            },
            "wrong_act_fitted_map": {
                donor_id: ep.transport_with(wrong, fitted)
                for donor_id, wrong in wrong_residuals.items()
            },
            "wrong_act_broken_map": {
                donor_id: {
                    map_id: ep.transport_with(wrong, broken_map)
                    for map_id, broken_map in broken_maps.items()
                }
                for donor_id, wrong in wrong_residuals.items()
            },
        }
        factorized_scores = map_factorized(
            vectors, lambda vector: score(decode_numpy(vector))["s"]
        )
        crossed = ep.materialize_crossed_factorials(factorized_scores)
        if crossed["unique_readout_count"] != 81:
            raise RuntimeError("factorized crossing must contain exactly 81 unique readouts")
        if crossed["logical_cell_count"] != 64:
            raise RuntimeError("factorized crossing must reconstruct exactly 64 logical cells")

        raw_records.append({
            "prompt_sha256": digest,
            "category": meta["category"],
            "layer": layer,
            "target_id": target_id,
            "target_source": "model_argmax",
            "target_derivation": target_derivation,
            "floor_scores": {
                "input_embedding_decoded": primary_floor["s"],
                "layer0_residual_decoded": sensitivity_floor["s"],
                "output_decoded": output_readout["s"],
            },
            "donor_assignments": donor_assignments,
            "map_draws": map_draws_by_layer[layer],
            "factorized_scores": factorized_scores,
        })

records, denominator_derivation = st.materialize_pilot_nta(raw_records)
if any("sensitivity_minus_primary" not in record["factorized_nta"] for record in records):
    raise RuntimeError("every materialized record must retain sensitivity_minus_primary")
NTA_MIN_DENOMINATOR = denominator_derivation["derived_value"]
STATISTICS_CODE_SHA256 = hashlib.sha256(
    pathlib.Path(st.__file__).read_bytes()
).hexdigest()
inference = st.build_pilot_inference(
    records,
    denominator_derivation,
    derivation_code_sha256=STATISTICS_CODE_SHA256,
    numpy_version=observed_runtime_versions["numpy"],
)
thresholds = inference["threshold_derivation"]
SPEC_MIN_EFFECT = thresholds.get("SPEC_MIN_EFFECT") if thresholds["available"] else None
INTERACTION_MIN_EFFECT = (
    thresholds.get("INTERACTION_MIN_EFFECT") if thresholds["available"] else None
)
REGISTRY["NTA_MIN_DENOMINATOR"]["declared_value"] = NTA_MIN_DENOMINATOR
REGISTRY["SPEC_MIN_EFFECT"]["declared_value"] = SPEC_MIN_EFFECT
REGISTRY["INTERACTION_MIN_EFFECT"]["declared_value"] = INTERACTION_MIN_EFFECT
pf.check_constant_registry(REGISTRY, pf.GATES)

descriptive = {
    "records": records,
    "factorization": {
        "unique_readouts_per_prompt_layer": 81,
        "logical_crossings_per_prompt_layer": 64,
        "donor_assignment_count": 8,
        "broken_map_draw_count": 8,
    },
}
print(
    f"pilot calculations complete for {len(records)} prompt-layer records; "
    f"derived NTA guard={NTA_MIN_DENOMINATOR}; "
    f"thresholds_available={thresholds['available']}"
)

## 7. Build, validate, and write the mode-correct aggregate

Pilot artifacts contain compact descriptive readouts, recomputable uncertainty, and threshold-derivation evidence but no scientific gate or decision. The derived vectors are not ratified confirmation thresholds until a later independent review and authorization.


In [ ]:
import datetime


def write_content_addressed(obj, prefix, out_dir=pathlib.Path(".")):
    canonical = json.dumps(obj, sort_keys=True, indent=2, ensure_ascii=False) + "\n"
    payload = canonical.encode("utf-8")
    digest = hashlib.sha256(payload).hexdigest()
    path = out_dir / f"{prefix}_{digest[:16]}.json"
    try:
        with path.open("xb") as handle:
            handle.write(payload)
    except FileExistsError:
        if path.read_bytes() != payload:
            raise RuntimeError(f"{path} exists with different bytes") from None
    if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        raise RuntimeError("content-addressed readback failed")
    return path, digest


def build_aggregate(*, run_mode, descriptive, denominator_derivation, inference):
    artifact = {
        "schema": SCHEMA,
        "artifact_type": "aggregate",
        "run_mode": run_mode,
        "run_id": RUN_ID,
        "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
        "evidence_class": "direct_runtime_measurement",
        "scope": "open_loop_observation_only",
        "model": {
            "repo_id": MODEL_ID,
            "revision": MODEL_REVISION,
            "n_layers": EXPECTED_MODEL_N_LAYERS,
            "d_model": EXPECTED_MODEL_D_MODEL,
        },
        "lens": {
            "repo_id": LENS_REPO,
            "revision": LENS_REVISION,
            "filename": LENS_FILE,
            "sha256": EXPECTED_LENS_SHA256,
            "source_layers": list(lens.source_layers),
            "d_model": lens.d_model,
        },
        "instrumentation": {"repo": JLENS_REPO_URL, "commit": installed_commit},
        "runtime": {
            "python": sys.version,
            "torch": torch.__version__,
            "packages": observed_runtime_versions,
            "cuda_runtime": torch.version.cuda,
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_total_vram_gib": env["vram_gib"],
            "install_schema": RUNTIME_INSTALL_SCHEMA,
            "install_spec_sha256": install_spec_sha256,
            "fresh_process_after_install": fresh_process_after_install,
            "torchvision_state": torchvision_state,
        },
        "stimulus_manifest": {
            "sha256": EXPECTED_MANIFEST_SHA256,
            "n_prompts": STAGE2B_N_PROMPTS,
        },
        "registry": pf.emit_registry_record(REGISTRY, pf.GATES),
        "disjointness": {
            "checked": True,
            "stage2b_manifest_sha256": EXPECTED_MANIFEST_SHA256,
            "overlap_count": 0,
            "anchor_present": False,
        },
        "preflight": {
            "pinned_identities_matched": True,
            "capacity_ok": True,
            "tensor_contracts_passed": True,
            "crossing_registry_checked": True,
        },
        "constants": {
            "min_denominator": NTA_MIN_DENOMINATOR,
            "guard_quantile": 0.05,
            "guard_quantile_method": "linear",
            "bootstrap_iterations": 20_000,
            "bootstrap_ci_level": 0.99,
            "bootstrap_quantile_method": "linear",
            "bootstrap_bit_generator": "PCG64",
        },
        "authorization": {
            "pilot_authorized": PILOT_AUTHORIZED,
            "pilot_protocol_ratified": PILOT_PROTOCOL_RATIFIED,
            "confirmatory_thresholds_ratified": THRESHOLDS_RATIFIED,
            "authorization_record_sha256": AUTHORIZATION_RECORD_SHA256,
            "authority": AUTHORIZATION_RECORD["decision"]["authority"],
            "authorized_at_utc": AUTHORIZATION_RECORD["decision"]["authorized_at_utc"],
            "instruction_sha256": AUTHORIZATION_RECORD["decision"]["instruction_sha256"],
            "notebook_sha256": AUTHORIZATION_RECORD["source"]["notebook_sha256"],
            "code_bundle_sha256": AUTHORIZATION_RECORD["source"]["code_bundle_sha256"],
        },
        "partition": {
            "n_prompts": partition["n_prompts"],
            "pilot_subset_sha256": partition["pilot_subset_sha256"],
            "pilot_view_sha256": EXPECTED_PILOT_VIEW_SHA256,
            "pilot_prompt_ids": [prompt["id"] for prompt in manifest["prompts"]],
            "pilot_prompt_sha256s": [prompt["sha256"] for prompt in manifest["prompts"]],
            "holdout_prompt_count": STAGE2B_N_PROMPTS - partition["n_prompts"],
            "holdout_accessed": False,
        },
        "design": {
            "selected_layers": SELECTED_LAYERS,
            "positions": POSITIONS,
            "top_k": TOP_K,
            "vocab_size": V,
            "model_n_layers": EXPECTED_MODEL_N_LAYERS,
            "primary_floor_id": "input_embedding_decoded",
            "sensitivity_floor_id": "layer0_residual_decoded",
            "donor_assignment_count": 8,
            "broken_map_draw_count": 8,
            "unique_readouts_per_prompt_layer": 81,
            "logical_crossings_per_prompt_layer": 64,
            "content_hash_method": "dtype-shape-bytes-sha256-v1",
        },
        "denominator_derivation": denominator_derivation,
        "descriptive": descriptive,
        "inference": inference,
        "retention": {
            "raw_activations_persisted": False,
            "full_logits_persisted": False,
            "raw_prompt_persisted": False,
        },
    }
    return artifact


aggregate = build_aggregate(
    run_mode=RUN_MODE,
    descriptive=descriptive,
    denominator_derivation=denominator_derivation,
    inference=inference,
)
validation_errors = []
validate_observation.validate_stage2b_aggregate(
    aggregate, pathlib.Path("in-memory.json"), "", validation_errors,
    expected_pilot_view=manifest,
    expected_source=TRUSTED_SOURCE_IDENTITIES,
)
if validation_errors:
    raise RuntimeError("aggregate validation failed: " + "; ".join(validation_errors))
artifact_path, artifact_sha256 = write_content_addressed(aggregate, "jspace_discrimination_s2b_pilot")
print(f"wrote {artifact_path} sha256={artifact_sha256}")


## Stage gate

Running the pilot requires one separately approved content-addressed authorization record bound to the exact canonical notebook, code bundle, and 20-prompt view. The denominator and threshold vectors are derived during the pilot under the ratified rules; they are not authorization inputs. Pilot authorization does not authorize the 180-prompt confirmation, artifact transfer, publication, Stage 3, or downstream integration.
